# Analytics: business-question and exploratory monitoring views

This compiled notebook creates the Analytics views used by the Analytics Dashboard and Genie. The notebook provides the documented execution sequence used by the Databricks job. It builds two groups of views:

- **6 official business-question views**, one per question Q1 to Q6 ([D-50](../../docs/decisions.md), D-51).
- **5 exploratory monitoring views**, for broader monitoring of project cohorts, project size, and implementing offices ([D-54, proposed](../../docs/decisions.md#proposed)). They add views. They never change Q1 to Q6.

All views use accepted Gold fact rows. Analytics creation remains separate from Gold modeling and from Analytics validation.

## Inputs

`dim_project`, `dim_region`, `dim_project_status`, `dim_flood_susceptibility`, `fact_project_snapshot`, `fact_region_population`, `fact_region_flood_exposure`. See [Gold model](../../docs/gold_model.md).

## Outputs

Official business-question views:

| View | Question |
| --- | --- |
| `analytics_regional_investment` | Q1. Regional investment distribution |
| `analytics_category_portfolio` | Q2. Infrastructure category portfolio |
| `analytics_project_delivery_status` | Q3. Project delivery and status |
| `analytics_population_equity` | Q4. Population-based equity |
| `analytics_flood_control_exposure` | Q5. Flood control versus flood exposure |
| `analytics_region_category` | Q6. Investment by area and type (D-27) |

Exploratory monitoring views:

| View | What it shows | One row per |
| --- | --- | --- |
| `analytics_infrastructure_year_trend` | How each infrastructure-year cohort looks in the current snapshot | `infra_year`, including `NULL` |
| `analytics_infrastructure_year_sector` | The sector mix of each cohort | `infra_year`, sector, and mapping status |
| `analytics_delivery_cohort` | The current status mix and delivery flags of each cohort | `infra_year`, status group, and status mapping status |
| `analytics_project_size_concentration` | Whether a region's budget sits in many small or a few large projects | Official region |
| `analytics_implementing_office_portfolio` | Where the observed project workload sits by implementing office | `implementing_office`, including `NULL` |

The views live in the `05-analytics` schema. Each view is rebuilt on every run with `CREATE OR REPLACE VIEW`. Views hold no state, so they need no `MERGE`.

## Rules used in every view

- Reads only rows where `is_current_snapshot = TRUE`, so each project counts once.
- Every regional view leaves out key `0` rows before any share or ratio. See [unknown and non-geographic records](../../docs/gold_model.md#unknown-and-non-geographic-records). Q3 to Q6 take their regions and coverage status from Q1, so the views agree.
- Every view carries `source_snapshot_id`, the DPWH snapshot behind its numbers ([D-29](../../docs/decisions.md)). Q5 also carries `mgb_source_snapshot_id`.
- Every table is named with its schema: Gold inputs as `` `03-gold`.table ``, views as `` `05-analytics`.view `` ([style guide](../../docs/style-guide.md)).
- An unmapped category or status stays `NULL` in its own row, with a column saying why ([D-37](../../docs/decisions.md)). No view writes `Unknown` or `Unmapped` text.
- `infra_year` is a **project cohort attribute** in the current snapshot. A cohort view compares the projects of each year as they look now. It never shows what the DPWH portfolio looked like in that year (D-54, proposed).
- The exploratory views reuse the Q3 status and delivery rules. They add no new status label and no second long-running or zero-progress definition.
- No view uses budget bands. Bands need their own decision.


In [ ]:
%sql
-- every table is named with its schema: gold inputs as `03-gold`.table, the views as `05-analytics`.view
USE CATALOG `buildabida-capstone`;

## 0. Check the snapshot and create the schema

### One current DPWH snapshot

**What:** stop unless the current rows of `fact_project_snapshot` hold exactly one DPWH snapshot, with no empty snapshot ID.

**Why:** every view reads `MAX(source_snapshot_id)` as the snapshot behind its numbers ([D-29](../../docs/decisions.md)). If current Gold mixed two snapshots, the views would add them together and still report one ID. This gate stops the run before any view is built. `10_validation_analytics` checks the same rule again before it validates.

**Expected result:** the `ASSERT_TRUE` returns `NULL`, and no error is raised.

In [ ]:
%sql
-- every view reads one dpwh snapshot, so stop before building anything if current gold mixes them
SELECT ASSERT_TRUE(
    COUNT(DISTINCT source_snapshot_id) = 1 AND COUNT_IF(source_snapshot_id IS NULL) = 0,
    'STOP: current Gold project rows span multiple DPWH snapshots. Rebuild Gold before analytics.'
)
FROM `03-gold`.fact_project_snapshot
WHERE is_current_snapshot = TRUE;

### Create the analytics schema

**What:** create `05-analytics` if it does not exist.

**Why:** the views are analytics outputs, not Gold tables. Their own schema keeps `03-gold` to the nine tables in the [Gold model](../../docs/gold_model.md) (build rule 6) and follows the numbered layer schemas.

**Expected result:** the schema exists. A rerun changes nothing.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS `05-analytics`
COMMENT 'Analytics views for the dashboard. Built only from validated 03-gold tables.';

## 1. Regional investment distribution (Q1)

**What:** build `analytics_regional_investment`. It answers: which Philippine regions receive the highest and lowest reported infrastructure investment and project counts? One row per official PSGC region, from the PSGC version that the current projects use. Key `0` holds Central Office, unmatched, and ambiguous projects. It is left out here, and `04_validation/10_validation_analytics` counts it separately.

**Why:** the view keeps these rules, so each project counts once and a source gap never looks like low investment.

- The view reads only rows where `is_current_snapshot = TRUE`.
- Budget means the reported project or contract budget, never payment ([D-28](../../docs/decisions.md)).
- A missing budget stays `NULL` and never becomes `0`. A budget of `0` stays `0`, because that is the value DPWH reported.
- Every official region keeps its row, even with no mapped project. `source_coverage_status` says how far to trust it:
  - `REPORTED`: the region has mapped DPWH projects.
  - `LIMITED_SOURCE_COVERAGE`: BARMM with mapped projects. DPWH coverage of BARMM is limited by the source.
  - `NO_MAPPED_PROJECT_DATA`: no current DPWH project maps to the region.
- Only `REPORTED` regions get a rank.

| Column | Definition |
| --- | --- |
| `project_count` | Distinct current projects mapped to the region. `0` when none maps to it. |
| `projects_with_budget` | Projects in `project_count` with a reported budget that is not `NULL` |
| `budget_coverage_pct` | `projects_with_budget` divided by `project_count`, times 100. `NULL` with no project. |
| `projects_with_coordinates` | Projects with a valid map point (`coordinate_status = 'VALID_PAIR'`) |
| `coordinate_coverage_pct` | `projects_with_coordinates` divided by `project_count`, times 100. The data model asks Q1 to show coordinate coverage. |
| `projects_matched_with_conflict` | Projects whose reported region name and map point disagree. The name region is kept ([D-31](../../docs/decisions.md)). This is the match coverage the data model asks for. |
| `total_reported_budget_pesos` | Sum of reported budgets. `NULL` when no project has a budget. |
| `avg_budget_per_project_pesos` | Total divided by `projects_with_budget`. Projects without a budget are not in the denominator. |
| `share_of_included_regions_budget_pct` | Region total divided by the total of every row in this view, times 100. Key `0` is not in the denominator. |
| `budget_rank` | 1 is the highest total. `REPORTED` regions only. Ties share a rank. |
| `project_count_rank` | 1 is the most projects. `REPORTED` regions only. Ties share a rank. |
| `project_vs_budget_rank_gap` | `project_count_rank` minus `budget_rank`. Above 0 means a better budget rank than project-count rank. Below 0 means the reverse. |
| `source_snapshot_id` | The DPWH snapshot behind every number ([D-29](../../docs/decisions.md)) |

These numbers describe the data. They do not show that a region is underfunded, overfunded, or treated unfairly.

**Expected result:** one row per official region. Only `REPORTED` regions have ranks. `04_validation/10_validation_analytics` checks every column against Gold.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_regional_investment (
    region_key COMMENT 'Official region. Joins to dim_region.region_key. Key 0 is never in this view.',
    region_name COMMENT 'Official PSGC region name.',
    region_short_name COMMENT 'Short region label for charts.',
    source_coverage_status COMMENT 'REPORTED, LIMITED_SOURCE_COVERAGE for BARMM, or NO_MAPPED_PROJECT_DATA. Only REPORTED regions are ranked.',
    project_count COMMENT 'Distinct current DPWH projects mapped to the region. 0 when none maps to it.',
    projects_with_budget COMMENT 'Current projects in the region with a reported budget that is not NULL.',
    budget_coverage_pct COMMENT 'projects_with_budget divided by project_count, times 100. NULL when the region has no project.',
    projects_with_coordinates COMMENT 'Projects in the region with a valid coordinate pair (coordinate_status VALID_PAIR).',
    coordinate_coverage_pct COMMENT 'projects_with_coordinates divided by project_count, times 100. NULL when the region has no project.',
    projects_matched_with_conflict COMMENT 'Projects whose reported region name differs from the region of their map point. The name region is kept (D-31).',
    total_reported_budget_pesos COMMENT 'Sum of reported project or contract budgets in pesos. Not actual payment. NULL when no project has a budget, never 0 for a missing value.',
    avg_budget_per_project_pesos COMMENT 'Total reported budget divided by projects_with_budget. NULL when no project has a budget.',
    share_of_included_regions_budget_pct COMMENT 'Region budget as a percent of the total across every row in this view. Key 0 is not in the denominator.',
    budget_rank COMMENT '1 means the highest reported budget. Ties share a rank. NULL when the region is not REPORTED.',
    project_count_rank COMMENT '1 means the most projects. Ties share a rank. NULL when the region is not REPORTED.',
    project_vs_budget_rank_gap COMMENT 'project_count_rank minus budget_rank. Above 0 means a better budget rank than project-count rank. NULL when not ranked.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Q1 regional investment. One row per official PSGC region with DPWH project count, reported budget, budget coverage, budget share, and ranks. Reported budget is not payment. Regions with no or limited DPWH coverage stay visible with a coverage status and get no rank. Key 0 is left out.'
AS
WITH current_projects AS (
    -- one current row per project. 10_validation_analytics stops the run if that breaks
    SELECT
        project_key,
        region_key,
        reported_budget_pesos,
        coordinate_status,
        region_match_status
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
        AND region_key <> 0
),

current_snapshot AS (
    SELECT MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

current_psgc_version AS (
    -- the psgc version behind the current region keys
    SELECT DISTINCT region.psgc_version
    FROM current_projects AS project
    INNER JOIN `03-gold`.dim_region AS region
        ON project.region_key = region.region_key
),

official_regions AS (
    -- every official region keeps a row, even one with no mapped project
    SELECT
        region.region_key,
        region.region_name,
        region.region_short_name,
        region.is_barmm
    FROM `03-gold`.dim_region AS region
    INNER JOIN current_psgc_version AS version
        ON region.psgc_version <=> version.psgc_version
    WHERE region.region_key <> 0
),

regional_projects AS (
    SELECT
        region_key,
        COUNT(DISTINCT project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN reported_budget_pesos IS NOT NULL THEN project_key END) AS projects_with_budget,
        COUNT(DISTINCT CASE WHEN coordinate_status = 'VALID_PAIR' THEN project_key END) AS projects_with_coordinates,
        COUNT(DISTINCT CASE WHEN region_match_status = 'MATCHED_WITH_CONFLICT' THEN project_key END)
            AS projects_matched_with_conflict,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos
    FROM current_projects
    GROUP BY ALL
),

regional_totals AS (
    SELECT
        region.region_key,
        region.region_name,
        region.region_short_name,
        COALESCE(projects.project_count, 0) AS project_count,
        COALESCE(projects.projects_with_budget, 0) AS projects_with_budget,
        COALESCE(projects.projects_with_coordinates, 0) AS projects_with_coordinates,
        COALESCE(projects.projects_matched_with_conflict, 0) AS projects_matched_with_conflict,
        -- no coalesce. a region without a budgeted project keeps a null budget, never 0
        projects.total_reported_budget_pesos,
        CASE
            WHEN COALESCE(projects.project_count, 0) = 0 THEN 'NO_MAPPED_PROJECT_DATA'
            -- readme and d-28: dpwh coverage of barmm is limited by the source
            WHEN region.is_barmm THEN 'LIMITED_SOURCE_COVERAGE'
            ELSE 'REPORTED'
        END AS source_coverage_status
    FROM official_regions AS region
    LEFT JOIN regional_projects AS projects
        ON region.region_key = projects.region_key
),

ranked AS (
    -- each status is ranked on its own, and only reported ranks are published
    SELECT
        regional_totals.*,
        RANK() OVER (
            PARTITION BY source_coverage_status
            ORDER BY total_reported_budget_pesos DESC NULLS LAST
        ) AS budget_rank_in_status,
        RANK() OVER (
            PARTITION BY source_coverage_status
            ORDER BY project_count DESC
        ) AS project_count_rank_in_status
    FROM regional_totals
)

SELECT
    region_key,
    region_name,
    region_short_name,
    source_coverage_status,
    project_count,
    projects_with_budget,
    ROUND(100.0 * projects_with_budget / NULLIF(project_count, 0), 2) AS budget_coverage_pct,
    projects_with_coordinates,
    ROUND(100.0 * projects_with_coordinates / NULLIF(project_count, 0), 2) AS coordinate_coverage_pct,
    projects_matched_with_conflict,
    total_reported_budget_pesos,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    ROUND(
        100.0 * total_reported_budget_pesos / NULLIF(SUM(total_reported_budget_pesos) OVER (), 0), 4
    ) AS share_of_included_regions_budget_pct,
    CASE WHEN source_coverage_status = 'REPORTED' THEN budget_rank_in_status END AS budget_rank,
    CASE WHEN source_coverage_status = 'REPORTED' THEN project_count_rank_in_status END AS project_count_rank,
    CASE
        WHEN source_coverage_status = 'REPORTED'
            THEN project_count_rank_in_status - budget_rank_in_status
    END AS project_vs_budget_rank_gap,
    current_snapshot.source_snapshot_id
FROM ranked
CROSS JOIN current_snapshot
ORDER BY budget_rank NULLS LAST, region_name;

## 2. Infrastructure category portfolio (Q2)

**What:** build `analytics_category_portfolio`. It answers: which infrastructure categories receive the largest share of reported budget, and how does flood control compare with other categories? One row per `standardized_sector`, `is_dpwh_flood_related`, and `infra_type_mapping_status` combination. `Multi-sector` splits into a row with a flood component and a row without one.

**Why:** a project with no usable category keeps its own row with `standardized_sector` `NULL` and `infra_type_mapping_status` saying why ([D-37](../../docs/decisions.md)), so no budget drops out of the shares. The view never writes `Unmapped` as a sector, because it would look like an approved category. A dashboard can label the `NULL` row at display time. Category does not depend on region, so this view counts every current project, including Central Office and unmatched projects on region key `0`. Its total is the full Gold budget, which is a little more than the Q1 total. The average divides by `projects_with_budget`, the same rule as Q1. Flood control has two shares, and both repeat on every row so a dashboard tile needs no filter:

- `flood_control_single_sector_share_pct` is the **strict** share. It counts only projects whose one sector is Flood Control and Drainage.
- `flood_control_share_of_portfolio_pct` is the **broad** share. It counts every project with a Flood Control and Drainage component ([D-39](../../docs/decisions.md)), including the whole budget of Multi-sector projects, because DPWH does not split their budget by component.

The real flood control amount sits between the two. Show both with a note, and never present one as exact.

**Expected result:** one row per category combination, including the `NULL` sector row. The budget shares add up to 100.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_category_portfolio (
    standardized_sector COMMENT 'Approved DPWH sector or Multi-sector. NULL when the project has no usable category; infra_type_mapping_status says why (D-37).',
    is_dpwh_flood_related COMMENT 'True when the project has any Flood Control and Drainage component (D-39), so Multi-sector projects can be true. NULL when the sector is NULL.',
    infra_type_mapping_status COMMENT 'Why the sector is set or empty, copied from dim_project.',
    project_count COMMENT 'Distinct current DPWH projects in the category.',
    projects_with_budget COMMENT 'Projects in the category with a reported budget that is not NULL.',
    total_reported_budget_pesos COMMENT 'Sum of reported project or contract budgets in pesos. Not actual payment. NULL when no project has a budget.',
    avg_budget_per_project_pesos COMMENT 'Total reported budget divided by projects_with_budget. NULL when no project has a budget.',
    share_of_total_budget_pct COMMENT 'Category budget as a percent of the total across every row in this view.',
    flood_control_share_of_portfolio_pct COMMENT 'Broad flood control share. Budget of every project with a Flood Control and Drainage component (D-39), including whole Multi-sector projects, as a percent of the portfolio. The same value on every row.',
    flood_control_single_sector_share_pct COMMENT 'Strict flood control share. Budget of projects whose only sector is Flood Control and Drainage, as a percent of the portfolio. The same value on every row.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Q2 category portfolio. One row per DPWH sector and mapping status with project count, reported budget, budget share, and the flood control share of the portfolio. Counts every current project, including region key 0, because category does not depend on region.'
AS
WITH current_projects AS (
    SELECT
        project_key,
        reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

current_snapshot AS (
    SELECT MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

category_totals AS (
    SELECT
        project.standardized_sector,
        project.is_dpwh_flood_related,
        project.infra_type_mapping_status,
        COUNT(DISTINCT fact.project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN fact.reported_budget_pesos IS NOT NULL THEN fact.project_key END) AS projects_with_budget,
        SUM(fact.reported_budget_pesos) AS total_reported_budget_pesos
    FROM current_projects AS fact
    -- left join keeps every current project in the portfolio (d-36)
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    GROUP BY ALL
)

SELECT
    standardized_sector,
    is_dpwh_flood_related,
    infra_type_mapping_status,
    project_count,
    projects_with_budget,
    total_reported_budget_pesos,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    ROUND(
        100.0 * total_reported_budget_pesos / NULLIF(SUM(total_reported_budget_pesos) OVER (), 0), 4
    ) AS share_of_total_budget_pct,
    ROUND(
        100.0 * SUM(CASE WHEN is_dpwh_flood_related THEN total_reported_budget_pesos ELSE 0 END) OVER ()
        / NULLIF(SUM(total_reported_budget_pesos) OVER (), 0),
        4
    ) AS flood_control_share_of_portfolio_pct,
    ROUND(
        100.0 * SUM(
            CASE WHEN standardized_sector = 'Flood Control and Drainage' THEN total_reported_budget_pesos ELSE 0 END
        ) OVER ()
        / NULLIF(SUM(total_reported_budget_pesos) OVER (), 0),
        4
    ) AS flood_control_single_sector_share_pct,
    current_snapshot.source_snapshot_id
FROM category_totals
CROSS JOIN current_snapshot
ORDER BY total_reported_budget_pesos DESC NULLS LAST;

## 3. Project delivery and status (Q3)

**What:** build `analytics_project_delivery_status`. It answers: which regions and categories have the highest share of ongoing, inactive, or long-running projects? One row per region, category, and `status_group`. Key `0` regions are left out.

**Why:**

- Long-running reuses Gold's `long_running_flag` ([D-32](../../docs/decisions.md)). It is true when more than 24 months pass from start to completion, or to the snapshot date without a completion date. The view does not set its own threshold.
- The flag is `NULL` for a project without a start date, and `zero_progress_flag` is `NULL` when progress is unresolved. So each rate divides by the projects where its flag is known, not by every project. The `_evaluated_count` columns show those denominators.
- `pct_of_region_sector_projects` answers "which regions and categories have the most ongoing or inactive work?". The statuses of one region and sector add up to 100. `pct_of_region_projects` shows how big each group is within its region.
- An unmapped sector or status stays `NULL` ([D-37](../../docs/decisions.md)). `infra_type_mapping_status` says why the sector is empty. `status_mapping_status` says why the status is empty: `NO_SOURCE_STATUS` when Gold has no status for the project (key `0`), or `NO_APPROVED_STATUS_RULE` when the source status, often a shifted value, has no approved mapping ([Gold model](../../docs/gold_model.md#unmapped-values-stay-visible)). The view never writes `Unknown` or `Unmapped` text.
- Every official region keeps a row, with the same `source_coverage_status` as Q1. A region with no mapped project has one row with `project_count` 0 and empty sector, status, and reason columns, so it never disappears.

**Reading the result:** a long-running or zero-progress project is not proof of delay. DPWH gives no reliable target completion date, so the repo does not label any project delayed ([Data model](../../docs/data-model.md#business-question-coverage-planned)). A high zero-progress rate can also mean progress was never reported, not that work stalled.

**Expected result:** one row per region, category, and status group, plus one row for each region with no mapped project. Within each region, `pct_of_region_projects` adds up to 100. Within each region and sector, `pct_of_region_sector_projects` adds up to 100.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_project_delivery_status (
    region_key COMMENT 'Official region. Joins to dim_region.region_key. Key 0 is never in this view.',
    region_name COMMENT 'Official PSGC region name.',
    region_short_name COMMENT 'Short region label for charts.',
    source_coverage_status COMMENT 'Coverage status from analytics_regional_investment. REPORTED, LIMITED_SOURCE_COVERAGE, or NO_MAPPED_PROJECT_DATA.',
    standardized_sector COMMENT 'Approved sector or Multi-sector. NULL when the project has no usable category (see infra_type_mapping_status) and on the row of a region with no mapped project.',
    infra_type_mapping_status COMMENT 'Why the sector is set or empty, copied from dim_project. NULL on the row of a region with no mapped project.',
    status_group COMMENT 'FINISHED, ACTIVE, or INACTIVE. NULL when the status is not mapped (see status_mapping_status) and on the row of a region with no mapped project.',
    status_mapping_status COMMENT 'MAPPED, NO_SOURCE_STATUS when Gold has no status for the project, or NO_APPROVED_STATUS_RULE when the source status has no approved mapping. NULL on the row of a region with no mapped project.',
    project_count COMMENT 'Distinct current projects in the group.',
    long_running_evaluated_count COMMENT 'Projects in the group with a long-running flag. A project without a start date has none.',
    long_running_count COMMENT 'Projects in the group where long_running_flag is true.',
    zero_progress_evaluated_count COMMENT 'Projects in the group with a zero-progress flag. A project with unresolved progress has none.',
    zero_progress_count COMMENT 'Projects in the group where zero_progress_flag is true.',
    pct_of_region_projects COMMENT 'Group projects as a percent of all projects in the region.',
    pct_of_region_sector_projects COMMENT 'Group projects as a percent of the projects in the same region and sector. The statuses of one region and sector add up to 100.',
    pct_long_running COMMENT 'long_running_count divided by long_running_evaluated_count, times 100.',
    pct_zero_progress COMMENT 'zero_progress_count divided by zero_progress_evaluated_count, times 100.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Q3 delivery and status. One row per region, category, and status group with project counts, the share of region projects, and long-running and zero-progress rates over projects where each flag is known. Every official region keeps a row. Key 0 is left out.'
AS
WITH current_projects AS (
    SELECT
        region_key,
        status_key,
        project_key,
        long_running_flag,
        zero_progress_flag
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
        AND region_key <> 0
),

delivery AS (
    SELECT
        fact.region_key,
        project.standardized_sector,
        project.infra_type_mapping_status,
        status.status_group,
        CASE
            WHEN fact.status_key = 0 THEN 'NO_SOURCE_STATUS'
            WHEN status.status_group IS NULL THEN 'NO_APPROVED_STATUS_RULE'
            ELSE 'MAPPED'
        END AS status_mapping_status,
        COUNT(DISTINCT fact.project_key) AS project_count,
        COUNT(fact.long_running_flag) AS long_running_evaluated_count,
        COUNT_IF(fact.long_running_flag = TRUE) AS long_running_count,
        COUNT(fact.zero_progress_flag) AS zero_progress_evaluated_count,
        COUNT_IF(fact.zero_progress_flag = TRUE) AS zero_progress_count
    FROM current_projects AS fact
    INNER JOIN `03-gold`.dim_project_status AS status
        ON fact.status_key = status.status_key
    INNER JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    GROUP BY ALL
),

region_rows AS (
    -- every official region from q1, so a region with no mapped project keeps a row
    SELECT
        region.region_key,
        region.region_name,
        region.region_short_name,
        region.source_coverage_status,
        delivery.standardized_sector,
        delivery.infra_type_mapping_status,
        delivery.status_group,
        delivery.status_mapping_status,
        COALESCE(delivery.project_count, 0) AS project_count,
        COALESCE(delivery.long_running_evaluated_count, 0) AS long_running_evaluated_count,
        COALESCE(delivery.long_running_count, 0) AS long_running_count,
        COALESCE(delivery.zero_progress_evaluated_count, 0) AS zero_progress_evaluated_count,
        COALESCE(delivery.zero_progress_count, 0) AS zero_progress_count,
        region.source_snapshot_id
    FROM `05-analytics`.analytics_regional_investment AS region
    LEFT JOIN delivery
        ON region.region_key = delivery.region_key
)

SELECT
    region_key,
    region_name,
    region_short_name,
    source_coverage_status,
    standardized_sector,
    infra_type_mapping_status,
    status_group,
    status_mapping_status,
    project_count,
    long_running_evaluated_count,
    long_running_count,
    zero_progress_evaluated_count,
    zero_progress_count,
    ROUND(
        100.0 * project_count / NULLIF(SUM(project_count) OVER (PARTITION BY region_key), 0), 2
    ) AS pct_of_region_projects,
    ROUND(
        100.0 * project_count
        / NULLIF(SUM(project_count) OVER (PARTITION BY region_key, standardized_sector), 0),
        2
    ) AS pct_of_region_sector_projects,
    ROUND(100.0 * long_running_count / NULLIF(long_running_evaluated_count, 0), 2) AS pct_long_running,
    ROUND(100.0 * zero_progress_count / NULLIF(zero_progress_evaluated_count, 0), 2) AS pct_zero_progress,
    source_snapshot_id
FROM region_rows
ORDER BY region_name, standardized_sector, status_group, infra_type_mapping_status, status_mapping_status;

## 4. Population-based equity (Q4)

**What:** build `analytics_population_equity`. It answers: which regions receive a larger or smaller share of investment than their share of population? One row per official region. Key `0` is left out.

**Why:**

- Population comes from `fact_region_population` where `is_primary_source = TRUE`, the 2024 Table C count (D-18, D-24). Only the newest primary year is read, so two years never add up.
- Region, budget, and coverage come from `analytics_regional_investment`, so Q1 and Q4 always agree.
- Both shares in the gap use the same regions. A region that is not `REPORTED`, such as BARMM, has population but no comparable DPWH budget. Comparing it would push every other gap down. So the investment share and `comparable_population_share_pct` cover the same comparable regions only: `REPORTED`, with a budget, and with a population. The gap and per-capita figures are `NULL` for the rest. `population_share_pct` still shows every region's share of the national population.
- A positive gap means the investment share is above the population share. A negative gap means it is below. These describe allocation differences, not fairness.
- Population is one proxy for need. It does not represent every form of infrastructure need ([Data model](../../docs/data-model.md#business-question-coverage-planned)).
- Since the issue #88 fix, NCR includes every City of Manila barangay. Region totals miss 4,789 people (0.004 percent): `Sawata`, Davao del Norte (4,152 people, Region XI) and `San Rafael`, City of Calaca, Batangas (637 people, Region IV-A) stay `UNMATCHED` in Silver. Their regions' per-capita figures run very slightly high. See [Gold model](../../docs/gold_model.md#fact_region_population).

**Expected result:** one row per official region. Investment shares and comparable population shares each add up to 100 across the comparable regions. Population shares add up to 100 across all regions.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_population_equity (
    region_key COMMENT 'Official region. Joins to dim_region.region_key. Key 0 is never in this view.',
    region_name COMMENT 'Official PSGC region name.',
    region_short_name COMMENT 'Short region label for charts.',
    source_coverage_status COMMENT 'Coverage status from analytics_regional_investment. Only REPORTED regions get a share, gap, and per-capita value.',
    total_reported_budget_pesos COMMENT 'Reported budget of the region from analytics_regional_investment. Not actual payment. NULL when no project has a budget.',
    population_reference_year COMMENT 'Census year of the primary population source.',
    population_count COMMENT 'Primary-source population of the region from fact_region_population.',
    investment_share_pct COMMENT 'Region budget as a percent of the budget of all comparable regions (REPORTED, with a budget and a population). NULL when not comparable.',
    population_share_pct COMMENT 'Region population as a percent of the population of all official regions.',
    comparable_population_share_pct COMMENT 'Region population as a percent of the population of all comparable regions. NULL when not comparable.',
    equity_gap_pp COMMENT 'investment_share_pct minus comparable_population_share_pct, in percentage points. NULL when not comparable.',
    investment_per_capita_pesos COMMENT 'Reported budget divided by population. NULL when not comparable.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Q4 population equity. One row per official region with reported budget, primary census population, investment and population shares over the same REPORTED regions, the equity gap in percentage points, and investment per capita. Key 0 is left out.'
AS
WITH primary_population AS (
    SELECT
        region_key,
        reference_year,
        population_count
    FROM `03-gold`.fact_region_population
    WHERE is_primary_source = TRUE
        AND region_key <> 0
    -- the newest primary census year only, so two years never add up
    QUALIFY reference_year = MAX(reference_year) OVER ()
),

combined AS (
    SELECT
        region.region_key,
        region.region_name,
        region.region_short_name,
        region.source_coverage_status,
        region.total_reported_budget_pesos,
        population.reference_year AS population_reference_year,
        population.population_count,
        -- one comparable set for both shares: reported coverage, a budget, and a population
        CASE
            WHEN region.source_coverage_status = 'REPORTED'
                AND region.total_reported_budget_pesos IS NOT NULL
                AND population.population_count IS NOT NULL
                THEN region.total_reported_budget_pesos
        END AS comparable_budget_pesos,
        CASE
            WHEN region.source_coverage_status = 'REPORTED'
                AND region.total_reported_budget_pesos IS NOT NULL
                AND population.population_count IS NOT NULL
                THEN population.population_count
        END AS comparable_population,
        region.source_snapshot_id
    FROM `05-analytics`.analytics_regional_investment AS region
    LEFT JOIN primary_population AS population
        ON region.region_key = population.region_key
),

shares AS (
    SELECT
        combined.*,
        ROUND(
            100.0 * comparable_budget_pesos / NULLIF(SUM(comparable_budget_pesos) OVER (), 0), 4
        ) AS investment_share_pct,
        ROUND(100.0 * population_count / NULLIF(SUM(population_count) OVER (), 0), 4) AS population_share_pct,
        ROUND(
            100.0 * comparable_population / NULLIF(SUM(comparable_population) OVER (), 0), 4
        ) AS comparable_population_share_pct
    FROM combined
)

SELECT
    region_key,
    region_name,
    region_short_name,
    source_coverage_status,
    total_reported_budget_pesos,
    population_reference_year,
    population_count,
    investment_share_pct,
    population_share_pct,
    comparable_population_share_pct,
    investment_share_pct - comparable_population_share_pct AS equity_gap_pp,
    ROUND(comparable_budget_pesos / NULLIF(comparable_population, 0), 2) AS investment_per_capita_pesos,
    source_snapshot_id
FROM shares
ORDER BY equity_gap_pp DESC NULLS LAST, region_name;

## 5. Flood control investment versus flood exposure (Q5)

**What:** build `analytics_flood_control_exposure`. It answers: how does flood control investment and project coverage compare with regional flood-risk exposure? One row per official region. Key `0` is left out.

**Why:** two separate flood ideas stay separate here, never merged into one flag:

- Flood control **investment** comes from `dim_project.is_dpwh_flood_related`, DPWH's own category. It says what kind of project it is.
- Flood **exposure** comes from `fact_region_flood_exposure`, the DENR-MGB hazard map. It says how much of a region's area is in a mapped flood level.

Exposure rules:

- Only the newest Gold exposure run is read, so older MGB snapshots never add up.
- Flood levels can overlap, so level areas may be added only when `level_area_additivity_status` is `ADDITIVE_WITHIN_TOLERANCE` ([Gold model](../../docs/gold_model.md#fact_region_flood_exposure)). So every region with exposure data shows the area and share of each level on its own. `total_susceptible_area_sqkm` and pesos per square kilometer appear only when the levels are additive.
- `exposure_area_status` says which case applies: `ADDITIVE_WITHIN_TOLERANCE`, `NON_ADDITIVE_CROSS_LEVEL_OVERLAP`, `NOT_EVALUATED` when no safe boundary exists, or `NO_EXPOSURE_DATA`.

Investment rules:

- Region, total budget, and coverage come from `analytics_regional_investment`.
- A region with mapped projects but no flood control project shows 0 projects and a 0 percent share. A region with no mapped DPWH project, such as BARMM, shows `NULL`, because the source cannot say.
- Flood control here is the **broad** D-39 flag: every project with a Flood Control and Drainage component, including the whole budget of Multi-sector projects. Q2 shows how close the strict share is nationally.

Project-point rules:

- Each flood control project is placed by the MGB level at its own map point, from `flood_match_status` ([Silver Table 15](../../docs/silver_project_flood_mapping.md)). Every project lands in exactly one of five groups: High or Very High, Low or Moderate, ambiguous, valid point outside every mapped zone, or no usable point.
- This works even where the area figures are blank, because it does not need a region boundary or added-up levels.
- `flood_control_points_classified_pct` says how much of the region the measure covers. A point outside mapped zones is **not** low risk; MGB simply has no polygon there. Read the high share only together with the classified percent.

**Reading the result:** these are comparative patterns. A low flood control share in a highly exposed region does not prove insufficient protection, and a high share does not prove adequacy ([Data model](../../docs/data-model.md#business-question-coverage-planned)). A region without exposure data is not a low-risk region.

**Expected result:** one row per official region. Every region with exposure data has per-level areas, even when the total is `NULL`. In every region with mapped projects, the five point groups add up to `flood_control_project_count`.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_flood_control_exposure (
    region_key COMMENT 'Official region. Joins to dim_region.region_key. Key 0 is never in this view.',
    region_name COMMENT 'Official PSGC region name.',
    region_short_name COMMENT 'Short region label for charts.',
    source_coverage_status COMMENT 'Coverage status from analytics_regional_investment. NO_MAPPED_PROJECT_DATA means the flood control figures are unknown, not zero.',
    flood_control_project_count COMMENT 'Current DPWH projects with a Flood Control and Drainage component, the broad D-39 flag. NULL when no DPWH project maps to the region.',
    flood_control_budget_pesos COMMENT 'Reported budget of those projects in pesos, including the whole budget of Multi-sector projects. Not actual payment. NULL when no flood control project has a budget.',
    total_reported_budget_pesos COMMENT 'Reported budget of all projects in the region from analytics_regional_investment.',
    flood_control_share_of_region_budget_pct COMMENT 'Flood control budget as a percent of the region budget. NULL when no DPWH project maps to the region.',
    exposure_area_status COMMENT 'ADDITIVE_WITHIN_TOLERANCE, NON_ADDITIVE_CROSS_LEVEL_OVERLAP, NOT_EVALUATED when no safe boundary exists, or NO_EXPOSURE_DATA.',
    low_area_sqkm COMMENT 'Area in the Low MGB flood level, in square kilometers.',
    moderate_area_sqkm COMMENT 'Area in the Moderate MGB flood level, in square kilometers.',
    high_area_sqkm COMMENT 'Area in the High MGB flood level, in square kilometers.',
    very_high_area_sqkm COMMENT 'Area in the Very High MGB flood level, in square kilometers.',
    high_share_of_region_area_pct COMMENT 'Share of the region area in the High level, from Gold.',
    very_high_share_of_region_area_pct COMMENT 'Share of the region area in the Very High level, from Gold.',
    total_susceptible_area_sqkm COMMENT 'Sum of the level areas. NULL unless exposure_area_status is ADDITIVE_WITHIN_TOLERANCE, because levels can overlap.',
    flood_control_pesos_per_sqkm_exposed COMMENT 'Flood control budget divided by total_susceptible_area_sqkm. NULL when the total is NULL or no DPWH project maps to the region.',
    flood_control_points_high_or_very_high COMMENT 'Flood control projects whose map point sits in one High or Very High MGB level. NULL when no DPWH project maps to the region.',
    flood_control_points_low_or_moderate COMMENT 'Flood control projects whose map point sits in one Low or Moderate MGB level. NULL when no DPWH project maps to the region.',
    flood_control_points_ambiguous COMMENT 'Flood control projects whose map point sits in more than one MGB level. NULL when no DPWH project maps to the region.',
    flood_control_points_outside_mapped_zones COMMENT 'Flood control projects with a valid map point outside every mapped MGB zone. Not low risk. NULL when no DPWH project maps to the region.',
    flood_control_points_no_usable_point COMMENT 'Flood control projects with a missing or invalid map point. NULL when no DPWH project maps to the region.',
    flood_control_points_classified_pct COMMENT 'Percent of flood control projects placed in one MGB level. Shows how much of the region the point measure covers.',
    flood_control_points_high_share_of_classified_pct COMMENT 'High or Very High projects as a percent of the projects placed in one level. NULL when none were placed.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind the flood control numbers (D-29).',
    mgb_source_snapshot_id COMMENT 'MGB snapshot behind the exposure numbers (D-29).'
)
COMMENT 'Q5 flood control versus exposure. One row per official region with flood control projects and budget, the share of the region budget, the area of each MGB flood level, a total only where levels may be added, and flood control projects grouped by the MGB level at their map point. Key 0 is left out.'
AS
WITH flood_control_by_region AS (
    SELECT
        fact.region_key,
        COUNT(DISTINCT fact.project_key) AS flood_control_project_count,
        SUM(fact.reported_budget_pesos) AS flood_control_budget_pesos,
        -- each project lands in exactly one point group. unmatched with a valid point means outside every mapped zone, not low risk
        COUNT(DISTINCT CASE
            WHEN fact.flood_match_status = 'MATCHED' AND level.flood_susceptibility_level IN ('High', 'Very High')
                THEN fact.project_key
        END) AS points_high_or_very_high,
        COUNT(DISTINCT CASE
            WHEN fact.flood_match_status = 'MATCHED' AND level.flood_susceptibility_level IN ('Low', 'Moderate')
                THEN fact.project_key
        END) AS points_low_or_moderate,
        COUNT(DISTINCT CASE WHEN fact.flood_match_status = 'AMBIGUOUS' THEN fact.project_key END) AS points_ambiguous,
        COUNT(DISTINCT CASE
            WHEN fact.flood_match_status = 'UNMATCHED' AND fact.coordinate_status = 'VALID_PAIR' THEN fact.project_key
        END) AS points_outside_mapped_zones
    FROM `03-gold`.fact_project_snapshot AS fact
    INNER JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    LEFT JOIN `03-gold`.dim_flood_susceptibility AS level
        ON fact.flood_susceptibility_key = level.flood_susceptibility_key
    WHERE fact.is_current_snapshot = TRUE
        AND fact.region_key <> 0
        AND project.is_dpwh_flood_related = TRUE
    GROUP BY ALL
),

latest_exposure_run AS (
    -- the newest gold exposure run, so older mgb snapshots never add up
    SELECT
        run_id,
        mgb_source_snapshot_id
    FROM `03-gold`.fact_region_flood_exposure
    QUALIFY ROW_NUMBER() OVER (ORDER BY source_load_ts DESC, run_id) = 1
),

region_exposure AS (
    SELECT
        exposure.region_key,
        -- one status per region. any level that is not additive blocks the total
        CASE
            WHEN COUNT_IF(exposure.level_area_additivity_status = 'NOT_EVALUATED') > 0 THEN 'NOT_EVALUATED'
            WHEN COUNT_IF(NOT exposure.level_area_additivity_status <=> 'ADDITIVE_WITHIN_TOLERANCE') > 0
                THEN 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP'
            ELSE 'ADDITIVE_WITHIN_TOLERANCE'
        END AS exposure_area_status,
        MAX(CASE WHEN level.flood_susceptibility_level = 'Low' THEN exposure.susceptible_area_sqkm END) AS low_area_sqkm,
        MAX(CASE WHEN level.flood_susceptibility_level = 'Moderate' THEN exposure.susceptible_area_sqkm END)
            AS moderate_area_sqkm,
        MAX(CASE WHEN level.flood_susceptibility_level = 'High' THEN exposure.susceptible_area_sqkm END) AS high_area_sqkm,
        MAX(CASE WHEN level.flood_susceptibility_level = 'Very High' THEN exposure.susceptible_area_sqkm END)
            AS very_high_area_sqkm,
        MAX(CASE WHEN level.flood_susceptibility_level = 'High' THEN exposure.share_of_region_area_pct END)
            AS high_share_of_region_area_pct,
        MAX(CASE WHEN level.flood_susceptibility_level = 'Very High' THEN exposure.share_of_region_area_pct END)
            AS very_high_share_of_region_area_pct,
        SUM(exposure.susceptible_area_sqkm) AS summed_level_area_sqkm
    FROM `03-gold`.fact_region_flood_exposure AS exposure
    INNER JOIN latest_exposure_run AS latest
        ON exposure.run_id = latest.run_id
    INNER JOIN `03-gold`.dim_flood_susceptibility AS level
        ON exposure.flood_susceptibility_key = level.flood_susceptibility_key
    WHERE exposure.flood_susceptibility_key <> 0
    GROUP BY ALL
),

combined AS (
    SELECT
        region.region_key,
        region.region_name,
        region.region_short_name,
        region.source_coverage_status,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(flood.flood_control_project_count, 0)
        END AS flood_control_project_count,
        flood.flood_control_budget_pesos,
        region.total_reported_budget_pesos,
        COALESCE(exposure.exposure_area_status, 'NO_EXPOSURE_DATA') AS exposure_area_status,
        exposure.low_area_sqkm,
        exposure.moderate_area_sqkm,
        exposure.high_area_sqkm,
        exposure.very_high_area_sqkm,
        exposure.high_share_of_region_area_pct,
        exposure.very_high_share_of_region_area_pct,
        CASE
            WHEN exposure.exposure_area_status = 'ADDITIVE_WITHIN_TOLERANCE' THEN exposure.summed_level_area_sqkm
        END AS total_susceptible_area_sqkm,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(flood.points_high_or_very_high, 0)
        END AS flood_control_points_high_or_very_high,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(flood.points_low_or_moderate, 0)
        END AS flood_control_points_low_or_moderate,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(flood.points_ambiguous, 0)
        END AS flood_control_points_ambiguous,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(flood.points_outside_mapped_zones, 0)
        END AS flood_control_points_outside_mapped_zones,
        -- whatever is left has no usable point: missing or invalid coordinates, or an invalid source row
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA' THEN COALESCE(
                flood.flood_control_project_count - flood.points_high_or_very_high - flood.points_low_or_moderate
                - flood.points_ambiguous - flood.points_outside_mapped_zones, 0
            )
        END AS flood_control_points_no_usable_point,
        region.source_snapshot_id,
        latest.mgb_source_snapshot_id
    FROM `05-analytics`.analytics_regional_investment AS region
    LEFT JOIN flood_control_by_region AS flood
        ON region.region_key = flood.region_key
    LEFT JOIN region_exposure AS exposure
        ON region.region_key = exposure.region_key
    LEFT JOIN latest_exposure_run AS latest
        ON TRUE
)

SELECT
    region_key,
    region_name,
    region_short_name,
    source_coverage_status,
    flood_control_project_count,
    flood_control_budget_pesos,
    total_reported_budget_pesos,
    CASE
        WHEN source_coverage_status <> 'NO_MAPPED_PROJECT_DATA' THEN ROUND(
            100.0 * COALESCE(flood_control_budget_pesos, 0) / NULLIF(total_reported_budget_pesos, 0), 4
        )
    END AS flood_control_share_of_region_budget_pct,
    exposure_area_status,
    low_area_sqkm,
    moderate_area_sqkm,
    high_area_sqkm,
    very_high_area_sqkm,
    high_share_of_region_area_pct,
    very_high_share_of_region_area_pct,
    total_susceptible_area_sqkm,
    CASE
        WHEN source_coverage_status <> 'NO_MAPPED_PROJECT_DATA' THEN ROUND(
            COALESCE(flood_control_budget_pesos, 0) / NULLIF(total_susceptible_area_sqkm, 0), 2
        )
    END AS flood_control_pesos_per_sqkm_exposed,
    flood_control_points_high_or_very_high,
    flood_control_points_low_or_moderate,
    flood_control_points_ambiguous,
    flood_control_points_outside_mapped_zones,
    flood_control_points_no_usable_point,
    ROUND(
        100.0 * (flood_control_points_high_or_very_high + flood_control_points_low_or_moderate)
        / NULLIF(flood_control_project_count, 0), 4
    ) AS flood_control_points_classified_pct,
    ROUND(
        100.0 * flood_control_points_high_or_very_high
        / NULLIF(flood_control_points_high_or_very_high + flood_control_points_low_or_moderate, 0), 4
    ) AS flood_control_points_high_share_of_classified_pct,
    source_snapshot_id,
    mgb_source_snapshot_id
FROM combined
ORDER BY very_high_area_sqkm DESC NULLS LAST, region_name;

## 6. Investment by area and type (Q6)

**What:** build `analytics_region_category`. It answers: how is each region's reported budget split across infrastructure types, and which regions hold the most of each type? One row per official region, `standardized_sector`, and `infra_type_mapping_status`. Key `0` is left out.

**Why:** [D-27](../../docs/decisions.md) names TWG-IDI as the main user, comparing reported investment **by area and type**. No other view can give budget for each region and sector, even when combined:

| View | Region | Sector | Budget | Why it cannot answer this |
| --- | --- | --- | --- | --- |
| Q1 | Yes | No | Yes | Budget per region, with all sectors mixed |
| Q2 | No | Yes | Yes | Budget per sector for the whole country |
| Q3 | Yes | Yes | No | Region and sector, but project counts only |
| Q4 | Yes | No | Yes | Budget per region only |
| Q5 | Yes | Flood only | Yes | One sector only |

A region's total and a sector's national total do not give the region's budget in that sector. Adding budget to Q3 instead was considered and rejected. Q3 is split by status, so a dashboard would have to add rows back together and rebuild every share, and adding percentages across rows gives wrong answers. A separate view keeps Q3 about delivery and gives D-27 one checked source with the shares ready.

- Regions and coverage come from Q1, so each region's sectors add up to its Q1 project count and budget.
- `share_of_region_budget_pct` answers "what does this region spend on?". The sectors of one region add up to 100.
- `share_of_sector_budget_pct` answers "where does this type of spending go?". The regions of one sector add up to 100.
- A project with no usable category keeps its own row with a `NULL` sector and `infra_type_mapping_status` saying why ([D-37](../../docs/decisions.md)). A region with no mapped project keeps one row with `project_count` 0, like Q3.
- The flood control row here is the **strict** single-sector Flood Control and Drainage. Multi-sector projects stay in `Multi-sector`, so no budget is counted twice.

**Reading the result:** these describe where reported budget sits. They do not show need, efficiency, or fairness.

**Expected result:** one row per region and sector, plus one row for each region with no mapped project.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_region_category (
    region_key COMMENT 'Official region. Joins to dim_region.region_key. Key 0 is never in this view.',
    region_name COMMENT 'Official PSGC region name.',
    region_short_name COMMENT 'Short region label for charts.',
    source_coverage_status COMMENT 'Coverage status from analytics_regional_investment. REPORTED, LIMITED_SOURCE_COVERAGE, or NO_MAPPED_PROJECT_DATA.',
    standardized_sector COMMENT 'Approved sector or Multi-sector. NULL when the project has no usable category (see infra_type_mapping_status) and on the row of a region with no mapped project.',
    infra_type_mapping_status COMMENT 'Why the sector is set or empty, copied from dim_project. NULL on the row of a region with no mapped project.',
    project_count COMMENT 'Distinct current DPWH projects in the region and sector.',
    projects_with_budget COMMENT 'Projects in the region and sector with a reported budget that is not NULL.',
    total_reported_budget_pesos COMMENT 'Sum of reported project or contract budgets in pesos. Not actual payment. NULL when no project has a budget.',
    avg_budget_per_project_pesos COMMENT 'Total reported budget divided by projects_with_budget. NULL when no project has a budget.',
    share_of_region_budget_pct COMMENT 'Sector budget as a percent of the region budget. The sectors of one region add up to 100.',
    share_of_sector_budget_pct COMMENT 'Region budget in this sector as a percent of the sector budget across every region in this view. Key 0 is not in the denominator.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Q6 investment by area and type (D-27). One row per official region, DPWH sector, and mapping status with project count, reported budget, the sector share of the region budget, and the region share of the sector budget. Every official region keeps a row. Key 0 is left out.'
AS
WITH current_projects AS (
    SELECT
        project_key,
        region_key,
        reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
        AND region_key <> 0
),

sector_totals AS (
    SELECT
        fact.region_key,
        project.standardized_sector,
        project.infra_type_mapping_status,
        COUNT(DISTINCT fact.project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN fact.reported_budget_pesos IS NOT NULL THEN fact.project_key END) AS projects_with_budget,
        SUM(fact.reported_budget_pesos) AS total_reported_budget_pesos
    FROM current_projects AS fact
    INNER JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    GROUP BY ALL
),

region_rows AS (
    -- every official region from q1, so a region with no mapped project keeps a row
    SELECT
        region.region_key,
        region.region_name,
        region.region_short_name,
        region.source_coverage_status,
        sector.standardized_sector,
        sector.infra_type_mapping_status,
        COALESCE(sector.project_count, 0) AS project_count,
        COALESCE(sector.projects_with_budget, 0) AS projects_with_budget,
        sector.total_reported_budget_pesos,
        region.source_snapshot_id
    FROM `05-analytics`.analytics_regional_investment AS region
    LEFT JOIN sector_totals AS sector
        ON region.region_key = sector.region_key
)

SELECT
    region_key,
    region_name,
    region_short_name,
    source_coverage_status,
    standardized_sector,
    infra_type_mapping_status,
    project_count,
    projects_with_budget,
    total_reported_budget_pesos,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    ROUND(
        100.0 * total_reported_budget_pesos
        / NULLIF(SUM(total_reported_budget_pesos) OVER (PARTITION BY region_key), 0),
        4
    ) AS share_of_region_budget_pct,
    ROUND(
        100.0 * total_reported_budget_pesos
        / NULLIF(SUM(total_reported_budget_pesos) OVER (PARTITION BY standardized_sector), 0),
        4
    ) AS share_of_sector_budget_pct,
    source_snapshot_id
FROM region_rows
ORDER BY region_name, total_reported_budget_pesos DESC NULLS LAST;

# Exploratory monitoring views

The five views below support broader monitoring and exploration ([D-54, proposed](../../docs/decisions.md#proposed)). They sit beside Q1 to Q6 and never change them. They follow the same rules: current-snapshot Gold only, one DPWH snapshot, a comment on every view and column, and `source_snapshot_id` on every row. `04_validation/10_validation_analytics` rebuilds each one from Gold.

**Reading `infra_year`:** `infra_year` is the infrastructure program year of a project. It is a **cohort attribute** of each project in the current snapshot. So these views can answer "how do the projects of different infrastructure-year cohorts look in the current snapshot?". They cannot answer "how did DPWH perform in 2016?", because Gold holds one current snapshot, not a picture of the portfolio in each past year. A newer cohort can have more active projects simply because its projects started later.

**Delivery rules:** the cohort and office views reuse the Q3 rules exactly:

- `status_group` comes from `dim_project_status` (D-40): `FINISHED`, `ACTIVE`, or `INACTIVE`. `ACTIVE` includes `On-Going` and `For Procurement`. The views write no new status label, such as `Completed` or `Ongoing`.
- Long-running reuses Gold's `long_running_flag` (D-32, D-49). It does not mean delayed.
- Each rate divides by the projects where its flag is known, shown by the `_evaluated_count` columns.
- `pct_zero_progress` keeps the Q3 definition, so it **includes** `For Procurement` projects. D-46 allows a zero-progress count that says it includes them. The cohort views say so in their comments and add `for_procurement_count`, so a high rate in a newer cohort can be read. A zero-progress project is not called stalled (D-46).

## 7. Infrastructure-year trend

**What:** build `analytics_infrastructure_year_trend`. It shows how the projects of each infrastructure-year cohort look in the current snapshot: how many there are, how large they are, and their current status and delivery flags. One row per `infra_year`.

**Why:**

- Year does not depend on region, so the view counts every current project, including Central Office and unmatched projects on region key `0`, like Q2.
- A project whose `infra_year` is `NULL` keeps its own row with `infra_year` `NULL`. The view writes no placeholder year. Gold sets `NULL` when the source value is not a whole year, and it does not say why, so the row has no reason column.
- The view keeps every year Gold publishes. It has no 2016 to 2026 filter. The validator lists the observed years with their counts, so shifted or outlier years stay visible.
- `finished_count`, `active_count`, and `inactive_count` use the approved status groups. `unmapped_status_count` holds status key `0` and source statuses without an approved rule. The four add up to `project_count`.
- The average divides by `projects_with_budget`, as in Q1. A missing budget stays `NULL`, and a reported 0 stays 0.

| Column | Definition |
| --- | --- |
| `project_count` | Distinct current projects in the cohort |
| `projects_with_budget` | Projects with a reported budget that is not `NULL` |
| `budget_coverage_pct` | `projects_with_budget` divided by `project_count`, times 100 |
| `total_reported_budget_pesos` | Sum of reported budgets. `NULL` when no project has a budget. |
| `avg_budget_per_project_pesos` | Total divided by `projects_with_budget` |
| `finished_count`, `active_count`, `inactive_count` | Projects in each approved status group |
| `unmapped_status_count` | Projects with no status group |
| `for_procurement_count` | Projects whose DPWH source status is `For Procurement`. They are part of `active_count`. |
| `long_running_evaluated_count`, `long_running_count`, `pct_long_running` | The Q3 long-running counts and rate |
| `zero_progress_evaluated_count`, `zero_progress_count`, `pct_zero_progress` | The Q3 zero-progress counts and rate. They include `For Procurement` projects. |
| `source_snapshot_id` | The DPWH snapshot behind every number ([D-29](../../docs/decisions.md)) |

**Reading the result:** compare cohorts as they look now. A higher active share in a newer cohort is expected, because those projects started later. Do not read a trend in DPWH performance from these rows.

**Expected result:** one row per observed `infra_year`, plus one `NULL` row when Gold has projects without a year. The rows add up to every current Gold project and budget.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_infrastructure_year_trend (
    infra_year COMMENT 'Infrastructure program year from dim_project.infra_year. A project cohort in the current snapshot, not a past state of the DPWH portfolio. NULL when the source value is not a whole year, and those projects keep this NULL row.',
    project_count COMMENT 'Distinct current DPWH projects in the cohort.',
    projects_with_budget COMMENT 'Projects in the cohort with a reported budget that is not NULL.',
    budget_coverage_pct COMMENT 'projects_with_budget divided by project_count, times 100.',
    total_reported_budget_pesos COMMENT 'Sum of reported project or contract budgets in pesos. Not actual payment. NULL when no project has a budget, never 0 for a missing value.',
    avg_budget_per_project_pesos COMMENT 'Total reported budget divided by projects_with_budget. NULL when no project has a budget.',
    finished_count COMMENT 'Projects whose approved status_group is FINISHED (D-40).',
    active_count COMMENT 'Projects whose approved status_group is ACTIVE (D-40). Includes On-Going and For Procurement projects.',
    inactive_count COMMENT 'Projects whose approved status_group is INACTIVE (D-40).',
    unmapped_status_count COMMENT 'Projects with no status group: status key 0, or a source status without an approved rule. The four status counts add up to project_count.',
    for_procurement_count COMMENT 'Projects whose DPWH source status is For Procurement. They are part of active_count, have not started work, and count in pct_zero_progress (D-46). Not stalled.',
    long_running_evaluated_count COMMENT 'Projects with a long-running flag. A project without a measurable start-to-end span has none (D-32, D-49).',
    long_running_count COMMENT 'Projects where long_running_flag is true. Long-running does not mean delayed.',
    pct_long_running COMMENT 'long_running_count divided by long_running_evaluated_count, times 100. The same rule as analytics_project_delivery_status.',
    zero_progress_evaluated_count COMMENT 'Projects with a zero-progress flag. A project with unresolved progress has none.',
    zero_progress_count COMMENT 'Projects where zero_progress_flag is true. Includes For Procurement projects (D-46), see for_procurement_count.',
    pct_zero_progress COMMENT 'zero_progress_count divided by zero_progress_evaluated_count, times 100. The same rule as analytics_project_delivery_status, so it includes For Procurement projects (D-46). Not a stalled rate.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Exploratory infrastructure-year trend (D-54, proposed). One row per infrastructure year, including NULL, with project count, reported budget, approved status groups, the For Procurement count, and long-running and zero-progress rates over projects where each flag is known. infra_year is a project cohort in the current snapshot, not a historical snapshot of the DPWH portfolio. Counts every current project, including region key 0.'
AS
WITH current_projects AS (
    SELECT
        project_key,
        status_key,
        reported_budget_pesos,
        long_running_flag,
        zero_progress_flag
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

current_snapshot AS (
    SELECT MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

cohort_totals AS (
    SELECT
        project.infra_year,
        COUNT(DISTINCT fact.project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN fact.reported_budget_pesos IS NOT NULL THEN fact.project_key END) AS projects_with_budget,
        SUM(fact.reported_budget_pesos) AS total_reported_budget_pesos,
        -- the approved d-40 status groups. no new status label is written
        COUNT(DISTINCT CASE WHEN status.status_group = 'FINISHED' THEN fact.project_key END) AS finished_count,
        COUNT(DISTINCT CASE WHEN status.status_group = 'ACTIVE' THEN fact.project_key END) AS active_count,
        COUNT(DISTINCT CASE WHEN status.status_group = 'INACTIVE' THEN fact.project_key END) AS inactive_count,
        COUNT(DISTINCT CASE WHEN status.status_group IS NULL THEN fact.project_key END) AS unmapped_status_count,
        -- d-46: for procurement stays in the zero-progress rate, and this count says how much of it they are
        COUNT(DISTINCT CASE WHEN status.source_status = 'For Procurement' THEN fact.project_key END)
            AS for_procurement_count,
        -- the q3 rules: each rate divides by the projects where its flag is known
        COUNT(fact.long_running_flag) AS long_running_evaluated_count,
        COUNT_IF(fact.long_running_flag = TRUE) AS long_running_count,
        COUNT(fact.zero_progress_flag) AS zero_progress_evaluated_count,
        COUNT_IF(fact.zero_progress_flag = TRUE) AS zero_progress_count
    FROM current_projects AS fact
    INNER JOIN `03-gold`.dim_project_status AS status
        ON fact.status_key = status.status_key
    -- left join keeps every current project (d-36). a null year keeps its own row
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    GROUP BY ALL
)

SELECT
    infra_year,
    project_count,
    projects_with_budget,
    ROUND(100.0 * projects_with_budget / NULLIF(project_count, 0), 2) AS budget_coverage_pct,
    total_reported_budget_pesos,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    finished_count,
    active_count,
    inactive_count,
    unmapped_status_count,
    for_procurement_count,
    long_running_evaluated_count,
    long_running_count,
    ROUND(100.0 * long_running_count / NULLIF(long_running_evaluated_count, 0), 2) AS pct_long_running,
    zero_progress_evaluated_count,
    zero_progress_count,
    ROUND(100.0 * zero_progress_count / NULLIF(zero_progress_evaluated_count, 0), 2) AS pct_zero_progress,
    current_snapshot.source_snapshot_id
FROM cohort_totals
CROSS JOIN current_snapshot
ORDER BY infra_year NULLS LAST;

## 8. Infrastructure-year sector mix

**What:** build `analytics_infrastructure_year_sector`. It shows whether the sector mix differs across infrastructure-year cohorts. One row per `infra_year`, `standardized_sector`, and `infra_type_mapping_status`.

**Why:**

- `share_of_year_budget_pct` answers "what does this cohort's budget go to?". Its denominator is the reported budget of the same `infra_year`, so the sectors of one year add up to 100. The `NULL` year is its own cohort.
- There is no share of a sector across years. The `NULL` year would sit in that denominator, so the known years would not add up to 100. A later version can add it with a documented denominator.
- Flood control here is the **strict** single sector, `standardized_sector = 'Flood Control and Drainage'`, as in Q6. Multi-sector projects stay in `Multi-sector`, so no budget counts twice. Q2 stays the source for the strict and broad national flood shares.
- A project with no usable category keeps a `NULL` sector, and `infra_type_mapping_status` says why ([D-37](../../docs/decisions.md)). The view writes no `Unknown`, `Unmapped`, or `Not categorized` text.
- The view counts every current project, including region key `0`.

**Reading the result:** the rows describe the current sector mix of each cohort. They do not show why a mix changed.

**Expected result:** one row per year and sector combination. Within each year, `share_of_year_budget_pct` adds up to 100.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_infrastructure_year_sector (
    infra_year COMMENT 'Infrastructure program year from dim_project.infra_year. A project cohort in the current snapshot, not a past state of the DPWH portfolio. NULL when the source value is not a whole year.',
    standardized_sector COMMENT 'Approved sector or Multi-sector (D-39). NULL when the project has no usable category; infra_type_mapping_status says why (D-37).',
    infra_type_mapping_status COMMENT 'Why the sector is set or empty, copied from dim_project.',
    project_count COMMENT 'Distinct current DPWH projects in the year and sector.',
    projects_with_budget COMMENT 'Projects in the year and sector with a reported budget that is not NULL.',
    total_reported_budget_pesos COMMENT 'Sum of reported project or contract budgets in pesos. Not actual payment. NULL when no project has a budget, never 0 for a missing value.',
    avg_budget_per_project_pesos COMMENT 'Total reported budget divided by projects_with_budget. NULL when no project has a budget.',
    share_of_year_budget_pct COMMENT 'Sector budget as a percent of the reported budget of the same infra_year. The sectors of one year add up to 100. The NULL year is its own cohort.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Exploratory infrastructure-year sector mix (D-54, proposed). One row per infrastructure year, DPWH sector, and mapping status with project count, reported budget, and the sector share of the year budget. infra_year is a project cohort in the current snapshot, not a historical snapshot of the DPWH portfolio. Flood control is the strict single sector. Counts every current project, including region key 0.'
AS
WITH current_projects AS (
    SELECT
        project_key,
        reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

current_snapshot AS (
    SELECT MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

sector_totals AS (
    SELECT
        project.infra_year,
        project.standardized_sector,
        project.infra_type_mapping_status,
        COUNT(DISTINCT fact.project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN fact.reported_budget_pesos IS NOT NULL THEN fact.project_key END) AS projects_with_budget,
        SUM(fact.reported_budget_pesos) AS total_reported_budget_pesos
    FROM current_projects AS fact
    -- left join keeps every current project (d-36)
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    GROUP BY ALL
)

SELECT
    infra_year,
    standardized_sector,
    infra_type_mapping_status,
    project_count,
    projects_with_budget,
    total_reported_budget_pesos,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    -- the denominator is the budget of the same year. a null year is its own partition
    ROUND(
        100.0 * total_reported_budget_pesos
        / NULLIF(SUM(total_reported_budget_pesos) OVER (PARTITION BY infra_year), 0),
        4
    ) AS share_of_year_budget_pct,
    current_snapshot.source_snapshot_id
FROM sector_totals
CROSS JOIN current_snapshot
ORDER BY infra_year NULLS LAST, total_reported_budget_pesos DESC NULLS LAST;

## 9. Delivery by cohort

**What:** build `analytics_delivery_cohort`. It shows whether older cohorts still hold active, inactive, long-running, or zero-progress projects. One row per `infra_year`, `status_group`, and `status_mapping_status`.

**Why:**

- `status_mapping_status` uses the Q3 rule exactly: `NO_SOURCE_STATUS` for status key `0`, `NO_APPROVED_STATUS_RULE` for a source status without an approved mapping, otherwise `MAPPED`. A `NULL` status group always has one of the two reasons, so unresolved statuses keep their own rows and never drop out.
- `pct_of_year_projects` divides by every project in the same `infra_year`, including the rows with no status group. The statuses of one year add up to 100. The dashboard does not rebuild this denominator.
- The long-running and zero-progress counts and rates are the Q3 rules. `for_procurement_count` shows how many projects of each row are still in procurement (D-46).
- The view counts every current project, including region key `0`.

**Reading the result:** older cohorts with remaining active or long-running projects may warrant more monitoring. These rows do not prove poor performance, and long-running does not mean delayed.

**Expected result:** one row per year, status group, and reason. Within each year, `pct_of_year_projects` adds up to 100, and the rows add up to the project count and flag denominators of `analytics_infrastructure_year_trend`.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_delivery_cohort (
    infra_year COMMENT 'Infrastructure program year from dim_project.infra_year. A project cohort in the current snapshot, not a past state of the DPWH portfolio. NULL when the source value is not a whole year.',
    status_group COMMENT 'Approved status group FINISHED, ACTIVE, or INACTIVE (D-40). NULL when the status is not mapped; status_mapping_status says why.',
    status_mapping_status COMMENT 'MAPPED, NO_SOURCE_STATUS when Gold has no status for the project (key 0), or NO_APPROVED_STATUS_RULE when the source status has no approved mapping. The same rule as analytics_project_delivery_status.',
    project_count COMMENT 'Distinct current DPWH projects in the year and status group.',
    pct_of_year_projects COMMENT 'project_count as a percent of every current project in the same infra_year, including projects with no status group. The rows of one year add up to 100.',
    for_procurement_count COMMENT 'Projects in the row whose DPWH source status is For Procurement. They have not started work and count in pct_zero_progress (D-46). Not stalled.',
    long_running_evaluated_count COMMENT 'Projects in the row with a long-running flag. A project without a measurable start-to-end span has none (D-32, D-49).',
    long_running_count COMMENT 'Projects in the row where long_running_flag is true. Long-running does not mean delayed.',
    pct_long_running COMMENT 'long_running_count divided by long_running_evaluated_count, times 100. The same rule as analytics_project_delivery_status.',
    zero_progress_evaluated_count COMMENT 'Projects in the row with a zero-progress flag. A project with unresolved progress has none.',
    zero_progress_count COMMENT 'Projects in the row where zero_progress_flag is true. Includes For Procurement projects (D-46), see for_procurement_count.',
    pct_zero_progress COMMENT 'zero_progress_count divided by zero_progress_evaluated_count, times 100. The same rule as analytics_project_delivery_status, so it includes For Procurement projects (D-46). Not a stalled rate.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Exploratory delivery by cohort (D-54, proposed). One row per infrastructure year, status group, and status mapping status with project count, the share of the year projects, the For Procurement count, and long-running and zero-progress rates over projects where each flag is known. infra_year is a project cohort in the current snapshot, not a historical snapshot of the DPWH portfolio. Counts every current project, including region key 0.'
AS
WITH current_projects AS (
    SELECT
        project_key,
        status_key,
        long_running_flag,
        zero_progress_flag
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

current_snapshot AS (
    SELECT MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

cohort_status AS (
    SELECT
        project.infra_year,
        status.status_group,
        -- the q3 rule, so both views give an unresolved status the same reason
        CASE
            WHEN fact.status_key = 0 THEN 'NO_SOURCE_STATUS'
            WHEN status.status_group IS NULL THEN 'NO_APPROVED_STATUS_RULE'
            ELSE 'MAPPED'
        END AS status_mapping_status,
        COUNT(DISTINCT fact.project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN status.source_status = 'For Procurement' THEN fact.project_key END)
            AS for_procurement_count,
        COUNT(fact.long_running_flag) AS long_running_evaluated_count,
        COUNT_IF(fact.long_running_flag = TRUE) AS long_running_count,
        COUNT(fact.zero_progress_flag) AS zero_progress_evaluated_count,
        COUNT_IF(fact.zero_progress_flag = TRUE) AS zero_progress_count
    FROM current_projects AS fact
    INNER JOIN `03-gold`.dim_project_status AS status
        ON fact.status_key = status.status_key
    -- left join keeps every current project (d-36). a null year keeps its own rows
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    GROUP BY ALL
)

SELECT
    infra_year,
    status_group,
    status_mapping_status,
    project_count,
    -- every project of the year is in the denominator, including unmapped statuses
    ROUND(
        100.0 * project_count / NULLIF(SUM(project_count) OVER (PARTITION BY infra_year), 0), 2
    ) AS pct_of_year_projects,
    for_procurement_count,
    long_running_evaluated_count,
    long_running_count,
    ROUND(100.0 * long_running_count / NULLIF(long_running_evaluated_count, 0), 2) AS pct_long_running,
    zero_progress_evaluated_count,
    zero_progress_count,
    ROUND(100.0 * zero_progress_count / NULLIF(zero_progress_evaluated_count, 0), 2) AS pct_zero_progress,
    current_snapshot.source_snapshot_id
FROM cohort_status
CROSS JOIN current_snapshot
ORDER BY infra_year NULLS LAST, status_group, status_mapping_status;

## 10. Project size concentration

**What:** build `analytics_project_size_concentration`. It compares whether a region's reported budget is spread over many smaller projects or sits in fewer larger ones. One row per official PSGC region. Key `0` is left out, and `04_validation/10_validation_analytics` counts it separately.

**Why:**

- Regions, `source_coverage_status`, project counts, the total budget, and the average come from `analytics_regional_investment`, so this view always agrees with Q1. Only `REPORTED` regions compare as normal regional coverage. BARMM stays visible as `LIMITED_SOURCE_COVERAGE`.
- The distribution uses no budget bands. Bands need their own approved decision. The view uses threshold-free figures instead:
  - `median_budget_per_project_pesos`: the median reported budget, `PERCENTILE_CONT(0.5)`, over projects with a reported budget that is not `NULL`.
  - `p90_budget_per_project_pesos`: the 90th percentile, `PERCENTILE_CONT(0.9)`, over the same projects. Both are exact, with linear interpolation between the two nearest budgets. They are never approximate.
  - `max_budget_per_project_pesos`: the largest reported budget in the region.
  - `largest_project_share_of_region_budget_pct`: the largest budget divided by the region total, times 100.
- A `NULL` budget is in no denominator. A reported 0 stays 0 and stays in the distribution, because that is the value DPWH reported. `For Procurement` projects report PHP 0 (D-46), so they pull the median down. `projects_with_zero_budget` shows how many there are.
- An average well above the median means a few large projects carry the region budget. These figures describe the data. They do not make either pattern good or bad.

**Expected result:** one row per official region, the same rows as Q1. A region with no mapped project has `NULL` median, p90, max, and largest-project share.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_project_size_concentration (
    region_key COMMENT 'Official region. Joins to dim_region.region_key. Key 0 is never in this view.',
    region_name COMMENT 'Official PSGC region name.',
    region_short_name COMMENT 'Short region label for charts.',
    source_coverage_status COMMENT 'Coverage status from analytics_regional_investment. Only REPORTED regions compare as normal regional coverage. LIMITED_SOURCE_COVERAGE marks BARMM.',
    project_count COMMENT 'Distinct current DPWH projects mapped to the region, from analytics_regional_investment.',
    projects_with_budget COMMENT 'Projects in the region with a reported budget that is not NULL. The median, p90, and average use only these projects.',
    projects_with_zero_budget COMMENT 'Projects in the region with a reported budget of exactly 0. They stay in the distribution as reported. For Procurement projects report 0 (D-46).',
    budget_coverage_pct COMMENT 'projects_with_budget divided by project_count, times 100. NULL when the region has no project.',
    total_reported_budget_pesos COMMENT 'Sum of reported project or contract budgets in pesos, from analytics_regional_investment. Not actual payment. NULL when no project has a budget.',
    avg_budget_per_project_pesos COMMENT 'Total reported budget divided by projects_with_budget, from analytics_regional_investment. NULL when no project has a budget.',
    median_budget_per_project_pesos COMMENT 'Exact median reported budget (PERCENTILE_CONT 0.5) over projects with a budget that is not NULL. A reported 0 is included. NULL when no project has a budget.',
    p90_budget_per_project_pesos COMMENT 'Exact 90th percentile reported budget (PERCENTILE_CONT 0.9) over projects with a budget that is not NULL. A reported 0 is included. NULL when no project has a budget.',
    max_budget_per_project_pesos COMMENT 'Largest reported project budget in the region. NULL when no project has a budget.',
    largest_project_share_of_region_budget_pct COMMENT 'max_budget_per_project_pesos divided by total_reported_budget_pesos, times 100. NULL when the total is NULL or 0.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Exploratory project size concentration (D-54, proposed). One row per official PSGC region with project count, reported budget, zero-budget projects, and the average, median, 90th percentile, and largest reported project budget, without budget bands. Regions and coverage come from analytics_regional_investment. Reported budget is not payment. Key 0 is left out.'
AS
WITH current_projects AS (
    SELECT
        region_key,
        reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
        AND region_key <> 0
),

budget_distribution AS (
    SELECT
        region_key,
        COUNT_IF(reported_budget_pesos = 0) AS projects_with_zero_budget,
        -- exact percentiles. a null budget is skipped, and a reported 0 stays in
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY reported_budget_pesos) AS median_budget_pesos,
        PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY reported_budget_pesos) AS p90_budget_pesos,
        MAX(reported_budget_pesos) AS max_budget_pesos
    FROM current_projects
    GROUP BY ALL
)

SELECT
    region.region_key,
    region.region_name,
    region.region_short_name,
    region.source_coverage_status,
    region.project_count,
    region.projects_with_budget,
    COALESCE(distribution.projects_with_zero_budget, 0) AS projects_with_zero_budget,
    region.budget_coverage_pct,
    region.total_reported_budget_pesos,
    region.avg_budget_per_project_pesos,
    ROUND(distribution.median_budget_pesos, 2) AS median_budget_per_project_pesos,
    ROUND(distribution.p90_budget_pesos, 2) AS p90_budget_per_project_pesos,
    distribution.max_budget_pesos AS max_budget_per_project_pesos,
    ROUND(
        100.0 * distribution.max_budget_pesos / NULLIF(region.total_reported_budget_pesos, 0), 4
    ) AS largest_project_share_of_region_budget_pct,
    region.source_snapshot_id
-- every official region from q1, so the regions and coverage status always agree
FROM `05-analytics`.analytics_regional_investment AS region
LEFT JOIN budget_distribution AS distribution
    ON region.region_key = distribution.region_key
ORDER BY region.total_reported_budget_pesos DESC NULLS LAST, region.region_name;

## 11. Implementing office portfolio

**What:** build `analytics_implementing_office_portfolio`. It describes where the observed project workload sits by implementing office. One row per `implementing_office`.

**Why:**

- `implementing_office` is the District Engineering Office that Silver resolves from the DPWH `deo` field and Gold copies. Each office has one row.
- A `NULL` office keeps its own row. It means the Gold value is unresolved or unavailable for this analysis. Gold does not say whether the office was missing or conflicting, so the view does not say either. It writes no `Missing`, `Conflict`, or `Unknown` text, and the `NULL` row gets no rank.
- `project_count_rank` and `budget_rank` use `RANK`, so ties share a rank. They rank real offices only.
- `region_count` counts the distinct official regions of an office's projects. Key `0` is not a region, so it is not counted. `sector_count` counts distinct non-`NULL` sectors, and `Multi-sector` counts as one sector.
- The median and average use only projects with a reported budget that is not `NULL`. The median is the exact `PERCENTILE_CONT(0.5)`.
- The long-running and zero-progress counts and rates are the Q3 rules. `pct_zero_progress` includes `For Procurement` projects (D-46).
- The view counts every current project, including region key `0`.

**Reading the result:** a high project count, a high budget, or a high long-running rate describes workload. It does not show office quality or performance.

**Expected result:** one row per office, plus one `NULL` row when Gold has projects without a resolved office. The rows add up to every current Gold project and budget.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_implementing_office_portfolio (
    implementing_office COMMENT 'Implementing District Engineering Office from dim_project. NULL when the Gold value is unresolved or unavailable. Gold does not say whether it was missing or conflicting. The NULL row is not a real office and gets no rank.',
    project_count COMMENT 'Distinct current DPWH projects of the office.',
    projects_with_budget COMMENT 'Projects of the office with a reported budget that is not NULL.',
    total_reported_budget_pesos COMMENT 'Sum of reported project or contract budgets in pesos. Not actual payment. NULL when no project has a budget, never 0 for a missing value.',
    avg_budget_per_project_pesos COMMENT 'Total reported budget divided by projects_with_budget. NULL when no project has a budget.',
    median_budget_per_project_pesos COMMENT 'Exact median reported budget (PERCENTILE_CONT 0.5) over projects with a budget that is not NULL. A reported 0 is included. NULL when no project has a budget.',
    region_count COMMENT 'Distinct official regions of the office projects. Region key 0 is not counted.',
    sector_count COMMENT 'Distinct sectors of the office projects that are not NULL. Multi-sector counts as one sector.',
    long_running_evaluated_count COMMENT 'Projects of the office with a long-running flag. A project without a measurable start-to-end span has none (D-32, D-49).',
    long_running_count COMMENT 'Projects of the office where long_running_flag is true. Long-running does not mean delayed.',
    pct_long_running COMMENT 'long_running_count divided by long_running_evaluated_count, times 100. The same rule as analytics_project_delivery_status.',
    zero_progress_evaluated_count COMMENT 'Projects of the office with a zero-progress flag. A project with unresolved progress has none.',
    zero_progress_count COMMENT 'Projects of the office where zero_progress_flag is true. Includes For Procurement projects (D-46).',
    pct_zero_progress COMMENT 'zero_progress_count divided by zero_progress_evaluated_count, times 100. The same rule as analytics_project_delivery_status, so it includes For Procurement projects (D-46). Not a stalled rate.',
    project_count_rank COMMENT '1 means the most projects. RANK, so ties share a rank. NULL for the NULL office row.',
    budget_rank COMMENT '1 means the highest reported budget. RANK, so ties share a rank. NULL for the NULL office row.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Exploratory implementing office portfolio (D-54, proposed). One row per implementing office, including NULL, with project count, reported budget, average and median budget, region and sector counts, long-running and zero-progress rates over projects where each flag is known, and ranks for real offices. Describes workload only, not office quality or performance. Counts every current project, including region key 0.'
AS
WITH current_projects AS (
    SELECT
        project_key,
        region_key,
        reported_budget_pesos,
        long_running_flag,
        zero_progress_flag
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

current_snapshot AS (
    SELECT MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

office_totals AS (
    SELECT
        project.implementing_office,
        COUNT(DISTINCT fact.project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN fact.reported_budget_pesos IS NOT NULL THEN fact.project_key END) AS projects_with_budget,
        SUM(fact.reported_budget_pesos) AS total_reported_budget_pesos,
        -- exact median. a null budget is skipped, and a reported 0 stays in
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY fact.reported_budget_pesos) AS median_budget_pesos,
        -- key 0 is not a region, so it is not counted
        COUNT(DISTINCT CASE WHEN fact.region_key <> 0 THEN fact.region_key END) AS region_count,
        COUNT(DISTINCT project.standardized_sector) AS sector_count,
        COUNT(fact.long_running_flag) AS long_running_evaluated_count,
        COUNT_IF(fact.long_running_flag = TRUE) AS long_running_count,
        COUNT(fact.zero_progress_flag) AS zero_progress_evaluated_count,
        COUNT_IF(fact.zero_progress_flag = TRUE) AS zero_progress_count
    FROM current_projects AS fact
    -- left join keeps every current project (d-36). a null office keeps its own row
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    GROUP BY ALL
),

ranked AS (
    -- real offices are ranked on their own, and the null office row never gets a rank
    SELECT
        office_totals.*,
        RANK() OVER (
            PARTITION BY implementing_office IS NULL
            ORDER BY project_count DESC
        ) AS project_count_rank_in_group,
        RANK() OVER (
            PARTITION BY implementing_office IS NULL
            ORDER BY total_reported_budget_pesos DESC NULLS LAST
        ) AS budget_rank_in_group
    FROM office_totals
)

SELECT
    implementing_office,
    project_count,
    projects_with_budget,
    total_reported_budget_pesos,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    ROUND(median_budget_pesos, 2) AS median_budget_per_project_pesos,
    region_count,
    sector_count,
    long_running_evaluated_count,
    long_running_count,
    ROUND(100.0 * long_running_count / NULLIF(long_running_evaluated_count, 0), 2) AS pct_long_running,
    zero_progress_evaluated_count,
    zero_progress_count,
    ROUND(100.0 * zero_progress_count / NULLIF(zero_progress_evaluated_count, 0), 2) AS pct_zero_progress,
    CASE WHEN implementing_office IS NOT NULL THEN project_count_rank_in_group END AS project_count_rank,
    CASE WHEN implementing_office IS NOT NULL THEN budget_rank_in_group END AS budget_rank,
    current_snapshot.source_snapshot_id
FROM ranked
CROSS JOIN current_snapshot
ORDER BY project_count_rank NULLS LAST, implementing_office;

# Summary

Built 11 views from current-snapshot Gold facts and dimensions: the 6 official business-question views, one per question, and the 5 exploratory monitoring views (D-54, proposed). The query below checks that every view returns rows. Analytics validation runs separately.

In [ ]:
%sql
SELECT 'analytics_regional_investment' AS view_name, COUNT(*) AS row_count FROM `05-analytics`.analytics_regional_investment
UNION ALL
SELECT 'analytics_category_portfolio', COUNT(*) FROM `05-analytics`.analytics_category_portfolio
UNION ALL
SELECT 'analytics_project_delivery_status', COUNT(*) FROM `05-analytics`.analytics_project_delivery_status
UNION ALL
SELECT 'analytics_population_equity', COUNT(*) FROM `05-analytics`.analytics_population_equity
UNION ALL
SELECT 'analytics_flood_control_exposure', COUNT(*) FROM `05-analytics`.analytics_flood_control_exposure
UNION ALL
SELECT 'analytics_region_category', COUNT(*) FROM `05-analytics`.analytics_region_category
UNION ALL
SELECT 'analytics_infrastructure_year_trend', COUNT(*) FROM `05-analytics`.analytics_infrastructure_year_trend
UNION ALL
SELECT 'analytics_infrastructure_year_sector', COUNT(*) FROM `05-analytics`.analytics_infrastructure_year_sector
UNION ALL
SELECT 'analytics_delivery_cohort', COUNT(*) FROM `05-analytics`.analytics_delivery_cohort
UNION ALL
SELECT 'analytics_project_size_concentration', COUNT(*) FROM `05-analytics`.analytics_project_size_concentration
UNION ALL
SELECT 'analytics_implementing_office_portfolio', COUNT(*) FROM `05-analytics`.analytics_implementing_office_portfolio;